# Wan 2.2 Video Generator — Colab T4

**All-in-one notebook** for video generation with ComfyUI + Wan 2.2 14B on a free T4 GPU.

| Workflow | Input | Output |
|----------|-------|--------|
| `video_wan_long_ultimate` | Image + text | Long video 20-30s (I2V + VACE) |
| `video_wan_t2v` | Text only | Text-to-video |
| `video_wan_clip` | Image + text | Short 3.4s clip for montage |
| `video_wan_v2v` | Video + text | Video-to-video restyling |
| `video_wan_talking` | Photo + audio | Talking head (FantasyTalking) |
| `video_wan_reels` | Photo folders | Reels from image sets |

**Run cells 1-6 in order**, then open the tunnel URL.

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Install ComfyUI + Custom Nodes
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Already cloned"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Custom nodes
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Already cloned"

!pip install -r ComfyUI-WanVideoWrapper/requirements.txt -q
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nInstalled!")

In [ ]:
#@title 3. Download Models (~28 GB, takes 5-10 min)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)

HF = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"

files = {
    # Wan 2.2 T2V 14B fp8 (~14 GB)
    f"{M}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors":
        f"{HF}/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
    # VACE module 14B (~5.7 GB)
    f"{M}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors":
        f"{HF}/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
    # FantasyTalking (~0.5 GB)
    f"{M}/diffusion_models/fantasytalking_fp16.safetensors":
        f"{HF}/fantasytalking_fp16.safetensors",
    # UMT5 XXL fp8 text encoder (~6.3 GB)
    f"{M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        f"{HF}/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    # VAE (~1.3 GB)
    f"{M}/vae/wan2.2_vae.safetensors":
        f"{HF}/wan2.2_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Already exists: {os.path.basename(dest)}")
    else:
        print(f"\nDownloading {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nAll models ready!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/*

In [ ]:
#@title 4. Download Workflows from Gist
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

workflows = [
    "video_wan_long_ultimate.json",
    "video_wan_t2v.json",
    "video_wan_clip.json",
    "video_wan_v2v.json",
    "video_wan_talking.json",
]

# Colab-specific workflows (fixed paths)
colab_workflows = {
    "video_wan_reels.json": "video_wan_reels_colab.json",
}

for wf in workflows:
    !wget -q -O "{WF_DIR}/{wf}" "{RAW}/{wf}"
    print(f"Downloaded: {wf}")

for local_name, gist_name in colab_workflows.items():
    !wget -q -O "{WF_DIR}/{local_name}" "{RAW}/{gist_name}"
    print(f"Downloaded: {gist_name} -> {local_name}")

print(f"\n{len(workflows) + len(colab_workflows)} workflows saved to {WF_DIR}")

In [ ]:
#@title 5. Upload Media (images / audio / video)
#@markdown Upload files for your workflows:
#@markdown - **Images** (.png, .jpg) for I2V, talking, clip workflows
#@markdown - **Audio** (.wav, .mp3) for talking workflow
#@markdown - **Video** (.mp4) for V2V workflow

from google.colab import files
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Saved: {path}")

print(f"\nFiles in input/: {os.listdir(INPUT_DIR)}")

In [ ]:
#@title 6. Launch ComfyUI
import subprocess, time, re

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Starting ComfyUI... (30s)")
time.sleep(30)

# Cloudflare tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI ready! Open: {url}")
    print(f"{'='*60}")
    print("\nLoad workflow: Menu -> Load -> pick any video_wan_* workflow")
else:
    print("Tunnel URL not found. ComfyUI running on port 8188.")

In [ ]:
#@title 7. Save Outputs to Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/wan"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("No outputs found yet. Generate some videos first!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Copied: {os.path.basename(v)}")
    print(f"\n{len(videos)} files saved to {dst}")

---
## Workflow Guide

### `video_wan_long_ultimate` — Long I2V (20-30s)
Upload image -> set prompt -> Queue. Uses VACE context windows for long coherent video.
- `num_frames`: 481=20s, 721=30s
- `blocks_to_swap`: 20 (default T4), increase to 30 if OOM

### `video_wan_t2v` — Text-to-Video
Text prompt only, no image needed. Pure generation from description.
- Default: 81 frames (3.4s). Increase for longer videos.

### `video_wan_clip` — Short Clip (3.4s)
Quick clip from optional start image. Ideal for generating multiple clips to montage.

### `video_wan_v2v` — Video-to-Video
Upload reference video -> restyle with text prompt. VACE preserves motion/structure.

### `video_wan_talking` — Talking Head
Upload portrait photo + audio file -> generates talking head video synced to speech.
- Use cell 7 below to generate speech audio via TTS
- `audio_scale`: lip sync strength (default works well)

### `video_wan_reels` — Reels from Folders
Loads images from folders. Originally for local use — on Colab, upload images and adjust folder paths in the workflow.

### Resolution Presets
| Size | Aspect | Use |
|------|--------|-----|
| 832x480 | 16:9 | Landscape |
| 480x832 | 9:16 | Portrait/Reels |
| 608x1080 | 9:16 | HD Portrait |

---
## Optional Tools
Run these cells as needed — they are independent of the main setup.

In [ ]:
#@title 8. Generate Speech Audio (TTS)
#@markdown Generate audio for the **talking** workflow using edge-tts.

text = "Hello, this is a demo of the talking head generator." #@param {type:"string"}
voice = "en-US-AriaNeural" #@param ["en-US-AriaNeural", "en-US-GuyNeural", "ru-RU-SvetlanaNeural", "ru-RU-DmitryNeural"]
output_name = "speech.wav" #@param {type:"string"}

!pip install edge-tts -q 2>/dev/null
!edge-tts --voice "{voice}" --text "{text}" --write-media /content/ComfyUI/input/{output_name}

# Preview
from IPython.display import Audio, display
display(Audio(f"/content/ComfyUI/input/{output_name}"))
print(f"\nSaved: /content/ComfyUI/input/{output_name}")
print("Select this file in the LoadAudio node in ComfyUI.")

In [ ]:
#@title 9. Enhance Prompt with Qwen2-VL (optional)
#@markdown Describe an uploaded image with Qwen2-VL 2B to generate a video prompt.
#@markdown Runs on CPU — takes 1-2 min.

image_name = "my_image.png" #@param {type:"string"}

!pip install transformers qwen-vl-utils -q 2>/dev/null

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print("Loading Qwen2-VL 2B on CPU...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

image_path = f"/content/ComfyUI/input/{image_name}"
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": f"file://{image_path}"},
        {"type": "text", "text": (
            "Describe this image in detail for use as a video generation prompt. "
            "Include: subject, action/pose, setting, lighting, colors, mood, "
            "camera angle. Write as one continuous paragraph, no bullet points."
        )}
    ]
}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")

print("Generating prompt...")
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256)

prompt = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)[0]

print(f"\n{'='*60}")
print("GENERATED PROMPT (copy to ComfyUI):")
print(f"{'='*60}")
print(prompt)

# Free memory
del model, processor
import gc; gc.collect()

In [ ]:
#@title 10. Montage: Combine Output Videos
#@markdown Concatenate all generated videos from ComfyUI output into one file.

import glob, os

output_dir = "/content/ComfyUI/output"
videos = sorted(glob.glob(f"{output_dir}/*.mp4"))

if not videos:
    print("No videos found in output/. Generate some videos first!")
else:
    print(f"Found {len(videos)} videos:")
    for i, v in enumerate(videos):
        print(f"  {i+1}. {os.path.basename(v)}")

    # Write file list for ffmpeg
    with open("/content/filelist.txt", "w") as f:
        for v in videos:
            f.write(f"file '{v}'\n")

    # Concatenate
    !ffmpeg -y -f concat -safe 0 -i /content/filelist.txt -c copy /content/montage.mp4 2>/dev/null

    print(f"\nMontage saved: /content/montage.mp4")
    !ls -lh /content/montage.mp4

    # Download
    from google.colab import files
    files.download("/content/montage.mp4")

---
## Coming Soon: AI Music Video Pipeline

**Concept:** Upload a music track and automatically generate a synced music video.

### How It Will Work
1. **Beat detection** (librosa) — analyze the track, find beats, segments, energy levels
2. **Prompt generation** — create a video prompt per segment based on lyrics/mood
3. **Clip generation** (ComfyUI API) — generate a short video clip per beat segment using `video_wan_clip`
4. **Assembly** (FFmpeg) — concatenate clips, sync to audio, add transitions

### Estimated Time
- ~45-90 min for a 3-min video on T4
- ~15-20 clips at 3.4s each

### Stack
- `librosa` — beat/onset detection, tempo analysis
- ComfyUI REST API — queue video_wan_clip per segment
- `ffmpeg` — concat + audio sync

**This will be a separate notebook: `colab_music_video.ipynb`**